### Esercizi SQL Esteso

È dato il seguente schema relazionale di un data mart

**INCASSO** (*IdSede, IdServizio, IdCategoriaAzienda, IdTempo*, #Consulenze, Incasso)

**SERVIZIO** (*IdServizio*, Servizio, TipologiaServizio)

**TEMPO** (*IdTempo*, Data, Mese, Bimestre, Trimestre, Quadrimestre, Semestre, Anno)

**SEDE-CONSULENTI** (*IdSede*, Sede, Città, Regione, AreaGeografica, #Consulenti)

**AZIENDA** (*IdAzienda*, CategoriaAzienda, SettoreCommerciale, Nazionalità)

### Punto 1

Dato lo schema logico precedente, considerare le seguenti query di interesse:

- Per ogni coppia (tipologia dei servizi, semestre), visualizzare l’incasso totale e il numero totale di consulenze, effettuate dai consulenti delle sedi site in Lombardia.

- Considerando solo le aziende italiane e tedesche, selezionare per ogni coppia (Regione delle
sedi dei consulenti, Servizio), l’incasso totale e il numero di consulenze effettuate,
separatamente per ogni anno.

- Considerando solo gli incassi del 2017, 2018, e 2019, per ogni coppia (Tipologia dei servizi,
nazionalità dell’azienda), visualizzare l’incasso semestrale e l’incasso semestrale medio per
consulenza.

Considerando le informazioni precedenti, definire una vista materializzata con **CREATE MATERIALIZED VIEW**, in modo da ridurre il tempo di risposta delle query di interesse sopra riportate. 

In [ ]:
CREATE MATERIALIZED VIEW ViewIncassi
BUILD IMMEDIATE
REFRESH FAST ON COMMIT
AS
SELECT Semestre, Anno, Servizio, TipoServizio, Regione, Nazionalità,
        SUM(Incasso) AS IncassoTot,
        SUM(#Consulenze) AS ConsulenzeTot
FROM INCASSO I 
JOIN TEMPO T ON T.IdTempo = I.IdTempo
JOIN SERVIZIO S ON S.IdServizio = I.IdServizio
JOIN "SEDE-CONSULENTI" SC ON SC.IdSede = I.IdSede
JOIN AZIENDA A ON A.IdAzienda = I.IdAzienda
GROUP BY Semestre, Anno, Servizio, TipoServizio, Regione, Nazionalità

### Punto 2

Si individui la combinazione minimale di attributi che costituisce un identificatore per la vista
materializzata ViewIncassi.

Combinazione minimale  = {**Semestre, Servizio, Regione, Nazionalità** }

### Punto 3

Si ipotizzi che che la vista materializzata Viewlncassi sia gestita senza utilizzare l’istruzione **CREATE MATERIALIZED VIEW**. La tabella derivata che realizza la vista materializzata è definita dalla seguente istruzione SQL:

CREATE TABLE ViewIncassi
(
    Servizio VARCHAR(20),

    TipologiaServizio VARCHAR(20),

    Nazionalità VARCHAR(20),

    Semestre VARCHAR(20),

    Anno VARCHAR(20),

    Regione VARCHAR(20),

    IncassoTot INTEGER

    CHECK (IncassoTot IS NOT NULL and IncassoTot > 0),

    NumConsulenzeTot INTEGER

    CHECK (NumConsulenzeTot IS NOT NULL and NumConsulenzeTot >0),

    PRIMARY KEY (Nazionalità, Semestre, Regione, Servizio)
)

Si scriva l’istruzione INSERT per il caricamento iniziale dei dati nella vista materializzata
ViewIncassi, utilizzando l’istruzione INSERT(SELECT…).

In [ ]:
INSERT INTO ViewIncassi (
    Semestre, Anno, Servizio, TipologiaServizio, Regione, Nazionalità, IncassoTot, NumConsulenzeTot
)

SELECT T.Semestre, T.Anno, S.Servizio, S.TipoServizio, SC.Regione, A.Nazionalità,
        SUM(Incasso),
        SUM(#Consulenze)
FROM INCASSO I 
JOIN TEMPO T ON T.IdTempo = I.IdTempo
JOIN SERVIZIO S ON S.IdServizio = I.IdServizio
JOIN "SEDE-CONSULENTI" SC ON SC.IdSede = I.IdSede
JOIN AZIENDA A ON A.IdAzienda = I.IdAzienda
GROUP BY T.Semestre, T.Anno, S.Servizio, S.TipoServizio, SC.Regione, A.Nazionalità

### Punto 4

Si ipotizzi che la gestione dell’aggiornamento della vista materializzata sia svolta mediante trigger.
Si scriva il trigger per propagare le modifiche alla vista materializzata ViewIncassi in caso di inserimento di un nuovo record nella tabella dei fatti INCASSO.

In [ ]:
CREATE OR REPLACE TRIGGER Trigger_IncassiVM
AFTER INSERT ON Incasso
FOR EACH ROW
DECLARE
    VarServizio, VarTipologiaServizio VARCHAR(20);
    VarRegione VARCHAR(20);
    VarNazionalità VARCHAR(4);
    VarSemestre VARCHAR(10);
    VarAnno INTEGER;
    N INTEGER;
BEGIN
    --Popoliamo ciascuna variabile
    SELECT Servizio, TipologiaServizio
    INTO VarServizio, VarTipologiaServizio
    FROM SERVIZIO
    WHERE IdServizio = :NEW.IdServizio;

    SELECT Regione
    INTO VarRegione
    FROM "SEDE-CONSULENTI"
    WHERE IdSede = :NEW.IdSede;

    SELECT Nazionalità
    INTO VarNazionalità
    FROM AZIENDA 
    WHERE IdAzienda = :NEW.IdAzienda;

    SELECT Semestre, Anno
    INTO VarSemestre, VarAnno
    FROM TEMPO 
    WHERE IdTempo = :NEW.IdTempo;

    SELECT COUNT (*)
    INTO N
    FROM ViewIncassi
    WHERE Semestre = VarSemestre
        AND Anno = VarAnno
        AND Regione = VarRegione
        AND Servizio = VarServizio
        AND TipologiaServizio = VarTipologiaServizio
        AND Nazionalità = VarNazionalità;
 -- N può essere 0 o 1 per ogni riga
 -- 1 significa che esiste un record della VM che va aggiornato
 -- 0 significa che si può inserire un nuovo record con i campi di :NEW
    IF N > 0 THEN
        UPDATE ViewIncassi
        SET IncassoTot = IncassoTot + :NEW.Incasso
            ConsulenzeTot = ConsulenzeTot + :NEW."#Consulenze"
            WHERE Semestre = VarSemestre
                AND Anno = VarAnno
                AND Regione = VarRegione
                AND Servizio = VarServizio
                AND TipologiaServizio = VarTipologiaServizio
                AND Nazionalità = VarNazionalità;
    ELSE 
        INSERT INTO ViewIncassi(
            Semestre, Anno, Servizio, TipologiaServizio, Regione, Nazionalità, IncassoTot, NumConsulenzeTot
        )
        VALUES (
            VarSemestre,
            VarAnno,
            VarServizio,
            VarTipologiaServizio,
            VarRegione,
            VarNazionalità,
            :NEW.Incasso,
            :NEW."#Consulenze"
        );
        
    END IF;

END IF;
 

### Punto 5

Si ipotizzi che la gestione dell’aggiornamento della vista materializzata sia svolta mediante trigger.
Si scriva il trigger per propagare l’aggiornamento (considerando la modalità di gestione del tempo
di tipo 1) del valore dell’attributo TipologiaServizio della tabella SERVIZIO alla vista materializzata
ViewIncassi.

**Hint**: Gestione del tempo di tipo 1 assume un replacing della dimensione anche retroattivo

In [ ]:

CREATE OR REPLACE Trigger_IncassiVM_UPD
AFTER UPDATE OF TipologiaServizio ON SERVIZIO
FOR EACH ROW 
BEGIN
    UPDATE ViewIncassi
    SET TipologiaServizio = :NEW.TipologiaServizio
    WHERE TipologiaServizio = :OLD.TipologiaServizio
END;

### Punto 6

La vista materializzata VM1 è creata con la seguente istruzione SQL:

CREATE MATERIALIZED VIEW
BUILD IMMEDIATE
REFRESH FAST ON DEMAND
ENABLE QUERY REWRITE
AS
    <Blocco A>  

Per la definizione del Blocco A, si veda il punto 1.
Scrivere le istruzioni che definiscono i MATERIALIZED VIEW LOG in Oracle necessari per l’aggiornamento automatico FAST della vista materializzata ViewIncassi. Si indichino tutti e soli i log necessari e all’interno di ogni definizione di log tutti e soli gli attributi necessari.

In [ ]:
CREATE MATERIALIZED VIEW LOG ON INCASSO
WITH ROWID SEQUENCE (
    IdTempo,
    IdServizio,
    IdSede,
    IdAzienda,
    Incasso,
    "#Consulenze"
)
INCLUDING NEW VAlUES;

CREATE MATERIALIZED VIEW LOG ON TEMPO 
WITH ROWID, SEQUENCE (
    IdTempo,
    Semestre,
    Anno
)
INCLUDING NEW VAlUES;

CREATE MATERIALIZED VIEW LOG ON SERVIZIO 
WITH ROWID, SEQUENCE (
    IdServizio,
    Servizio, 
    TipologiaServizio
)
INCLUDING NEW VAlUES;

CREATE MATERIALIZED VIEW LOG ON "SEDE-CONSULENTI"
WITH ROWID, SEQUENCE (
    IdSede,
    Regione
)
INCLUDING NEW VAlUES;

CREATE MATERIALIZED VIEW LOG ON AZIENDA 
WITH ROWID, SEQUENCE (
    IdAzienda,
    Nazionalità
)
INCLUDING NEW VAlUES;